In [60]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda:0")

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 380.28it/s]


In [147]:
df = pd.read_csv("/home/renile-ai/AI/Test-Repo/prices_selected_2015_2025.csv")
df.drop(columns=['فلفل الوان',"فراوله","برتقال ابو سرة","مانجو زبدية"], inplace=True)
df.head()

,date,طماطم
0,2015-01-01,0.90
1,2015-01-02,1.00
2,2015-01-03,0.90
3,2015-01-04,1.10
4,2015-01-05,1.15


In [148]:
df["index"] = "tomato"
df.head()

,date,طماطم,index
0,2015-01-01,0.90,tomato
1,2015-01-02,1.00,tomato
2,2015-01-03,0.90,tomato
3,2015-01-04,1.10,tomato
4,2015-01-05,1.15,tomato


In [149]:
train_df = df.iloc[2922:3653] 
test_df = df.iloc[3653:] 
test_df.head()

,date,طماطم,index
3653,2025-01-01,5.00,tomato
3654,2025-01-02,4.75,tomato
3655,2025-01-03,5.00,tomato
3656,2025-01-04,4.75,tomato
3657,2025-01-05,5.00,tomato


In [150]:
test_df.tail(5)

,date,طماطم,index
3738,2025-03-27,4.5,tomato
3739,2025-03-28,5.5,tomato
3740,2025-03-29,5.5,tomato
3741,2025-03-30,7.0,tomato
3742,2025-03-31,8.0,tomato


In [151]:
train_df.tail()

,date,طماطم,index
3648,2024-12-27,5.00,tomato
3649,2024-12-28,5.25,tomato
3650,2024-12-29,5.50,tomato
3651,2024-12-30,5.50,tomato
3652,2024-12-31,5.50,tomato


In [152]:
train_df.head()

,date,طماطم,index
2922,2023-01-01,5.0,tomato
2923,2023-01-02,5.0,tomato
2924,2023-01-03,5.0,tomato
2925,2023-01-04,4.0,tomato
2926,2023-01-05,5.0,tomato


In [153]:
# Generate predictions
pred_df = pipeline.predict_df(
    df=train_df,
    prediction_length=90,  # Number of steps to forecast
    quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],  # Quantiles for probabilistic forecast
    id_column="index",  # Column identifying different time series
    timestamp_column="date",  # Column with datetime information
    target="طماطم",  # Column(s) with time series values to predict
)

In [154]:
pred_df.head()

,index,date,target_name,predictions,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
0,tomato,2025-01-01,طماطم,5.518172,5.061538,5.222683,5.337307,5.437482,5.518172,5.605493,5.698133,5.832525,6.070617
1,tomato,2025-01-02,طماطم,5.470466,4.868587,5.085691,5.236251,5.359022,5.470466,5.586501,5.710149,5.872340,6.145754
2,tomato,2025-01-03,طماطم,5.404456,4.727961,4.959779,5.123754,5.267568,5.404456,5.545308,5.694171,5.886826,6.202899
3,tomato,2025-01-04,طماطم,5.341153,4.581178,4.846772,5.034519,5.193000,5.341153,5.492439,5.656094,5.869041,6.224387
4,tomato,2025-01-05,طماطم,5.283809,4.495916,4.771586,4.963700,5.128856,5.283809,5.441031,5.615472,5.853179,6.251393


In [155]:
pred_df.tail()

,index,date,target_name,predictions,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
85,tomato,2025-03-27,طماطم,5.168211,3.811017,4.244364,4.572385,4.872684,5.168211,5.496757,5.891014,6.471589,7.419956
86,tomato,2025-03-28,طماطم,5.292324,3.884501,4.332257,4.674677,4.979213,5.292324,5.645830,6.051373,6.629287,7.565990
87,tomato,2025-03-29,طماطم,5.409070,3.916744,4.395615,4.755583,5.079587,5.409070,5.788174,6.218328,6.818930,7.790360
88,tomato,2025-03-30,طماطم,5.447300,3.945024,4.436543,4.796112,5.117316,5.447300,5.827648,6.267519,6.873037,7.816618
89,tomato,2025-03-31,طماطم,5.498401,3.935055,4.431249,4.815835,5.162245,5.498401,5.870632,6.306963,6.916526,7.891795


In [157]:
test_df.tail()

,date,طماطم,index
3738,2025-03-27,4.5,tomato
3739,2025-03-28,5.5,tomato
3740,2025-03-29,5.5,tomato
3741,2025-03-30,7.0,tomato
3742,2025-03-31,8.0,tomato


In [156]:
len(pred_df),len(test_df)

(90, 90)

In [141]:
test_df = test_df.iloc[:15]
test_df.tail()

,date,طماطم,index
3663,2025-01-11,5.0,tomato
3664,2025-01-12,5.0,tomato
3665,2025-01-13,5.0,tomato
3666,2025-01-14,5.0,tomato
3667,2025-01-15,5.0,tomato


In [142]:
test_df.head()

,date,طماطم,index
3653,2025-01-01,5.00,tomato
3654,2025-01-02,4.75,tomato
3655,2025-01-03,5.00,tomato
3656,2025-01-04,4.75,tomato
3657,2025-01-05,5.00,tomato


In [143]:
# RMSE, MAE and MAPE between actual prices and the 0.5 (median) predictions
n = len(pred_df)
actual = test_df["طماطم"].iloc[:n].to_numpy()
pred_05 = pred_df["0.5"].to_numpy()

rmse_05 = np.sqrt(np.mean((actual - pred_05) ** 2))
mae_05 = np.mean(np.abs(actual - pred_05))
mape_05 = np.mean(np.abs((actual - pred_05) / actual)) * 100

print(f"0.5 predictions -> RMSE: {rmse_05:.4f}, MAE: {mae_05:.4f}, MAPE: {mape_05:.2f}%")

0.5 predictions -> RMSE: 0.3717, MAE: 0.2904, MAPE: 6.00%


In [144]:
# RMSE, MAE and MAPE between actual prices and the 0.1 (lower quantile) predictions
pred_01 = pred_df["0.1"].to_numpy()

rmse_01 = np.sqrt(np.mean((actual - pred_01) ** 2))
mae_01 = np.mean(np.abs(actual - pred_01))
mape_01 = np.mean(np.abs((actual - pred_01) / actual)) * 100

print(f"0.1 predictions -> RMSE: {rmse_01:.4f}, MAE: {mae_01:.4f}, MAPE: {mape_01:.2f}%")

0.1 predictions -> RMSE: 0.6293, MAE: 0.5543, MAPE: 11.14%


In [145]:
# RMSE, MAE and MAPE between actual prices and the 0.4 (lower quantile) predictions
pred_04 = pred_df["0.4"].to_numpy()

rmse_04 = np.sqrt(np.mean((actual - pred_04) ** 2))
mae_04 = np.mean(np.abs(actual - pred_04))
mape_04 = np.mean(np.abs((actual - pred_04) / actual)) * 100

print(f"0.4 predictions -> RMSE: {rmse_04:.6f}, MAE: {mae_04:.6f}, MAPE: {mape_04:.2f}%")

0.4 predictions -> RMSE: 0.287291, MAE: 0.218091, MAPE: 4.51%


In [146]:
# RMSE, MAE and MAPE between actual prices and the 0.9 (upper quantile) predictions
pred_09 = pred_df["0.9"].to_numpy()

rmse_09 = np.sqrt(np.mean((actual - pred_09) ** 2))
mae_09 = np.mean(np.abs(actual - pred_09))
mape_09 = np.mean(np.abs((actual - pred_09) / actual)) * 100

print(f"0.9 predictions -> RMSE: {rmse_09:.2f}, MAE: {mae_09:.2f}, MAPE: {mape_09:.2f}%")

0.9 predictions -> RMSE: 1.38, MAE: 1.37, MAPE: 27.84%
